# E2E Wrapper
Notebook này chỉ load checkpoint object, attribute, relation rồi wrap chúng thành một E2E inference flow.

In [ ]:
import importlib
import json
import random
import sys
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, Optional, Sequence, Tuple

import torch


def resolve_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "configs").exists() and (current / "src").exists():
            return current
        current = current.parent
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án chứa configs/ và src/.")


ROOT_DIR = resolve_project_root(Path.cwd())
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

for module_name in [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]:
    del sys.modules[module_name]
importlib.invalidate_caches()

print(f"Resolved project root: {ROOT_DIR}")
print(f"Sys path head: {sys.path[:3]}")
print(f"Src exists: {(ROOT_DIR / 'src').exists()} | Data package exists: {(ROOT_DIR / 'src' / 'data').exists()}")

from src.data.attribute_dataset import build_attribute_datasets
from src.data.object_dataset import build_object_datasets
from src.data.relation_dataset import build_relation_datasets
from src.models.attribute import AttributeClassifier
from src.models.caption import CaptionGenerator
from src.models.e2e import VisualGenomeE2EModel
from src.models.object import ObjectClassifier
from src.models.relation import RelationClassifier
from src.utils.config_loader import load_config, load_task_configs
from src.utils.memory import cleanup_cuda_memory


def resolve_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    candidate_text = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Không tìm thấy path hợp lệ trong: {candidate_text}")


def load_vocab_file(vocab_path: Path) -> Dict[str, int]:
    if not vocab_path.exists():
        raise FileNotFoundError(f"Không tìm thấy vocabulary file: {vocab_path}")
    with open(vocab_path, "r", encoding="utf-8") as f:
        return json.load(f)


def inverse_vocab(vocab: Dict[str, int]) -> Dict[int, str]:
    return {int(index): label for label, index in vocab.items()}


def find_checkpoint_file(checkpoint_root: Path, filename_prefixes: Sequence[str]) -> Optional[Path]:
    if not checkpoint_root.exists():
        return None

    candidates = [
        path for path in checkpoint_root.rglob("*.pth")
        if any(path.name.startswith(prefix) for prefix in filename_prefixes)
    ]

    if not candidates:
        return None

    def score(path: Path) -> Tuple[float, float]:
        try:
            checkpoint = torch.load(path, map_location="cpu")
            metadata = checkpoint.get("metadata", {}) if isinstance(checkpoint, dict) else {}
            metric = metadata.get("metric", checkpoint.get("metric", float("-inf")) if isinstance(checkpoint, dict) else float("-inf"))
        except Exception:
            metric = float("-inf")
        return float(metric), float(path.stat().st_mtime)

    return max(candidates, key=score)


def checkpoint_uses_backbone(checkpoint_path: Path) -> bool:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    if not isinstance(state_dict, dict):
        return False
    return any(key.startswith("encoder.") for key in state_dict.keys())


def to_batch_tensor(sample: Dict[str, torch.Tensor], keys: Sequence[str], device: str) -> torch.Tensor:
    for key in keys:
        value = sample.get(key)
        if isinstance(value, torch.Tensor):
            return value.unsqueeze(0).to(device)
    raise KeyError(f"Không tìm thấy tensor trong sample với các key: {keys}")


def top_class_predictions(logits: torch.Tensor, idx_to_label: Dict[int, str], top_k: int = 5):
    probabilities = torch.softmax(logits, dim=1)[0]
    top_k = min(top_k, probabilities.numel())
    values, indices = torch.topk(probabilities, k=top_k)
    return [
        (idx_to_label.get(int(index), str(int(index))), float(value))
        for value, index in zip(values, indices)
    ]


def top_attribute_predictions(logits: torch.Tensor, idx_to_label: Dict[int, str], top_k: int = 5, threshold: float = 0.5):
    probabilities = torch.sigmoid(logits)[0]
    top_k = min(top_k, probabilities.numel())
    values, indices = torch.topk(probabilities, k=top_k)
    decoded = [
        (idx_to_label.get(int(index), str(int(index))), float(value))
        for value, index in zip(values, indices)
    ]
    active = [item for item in decoded if item[1] >= threshold]
    return active if active else decoded[:1]


BASE_CONFIG_PATH = ROOT_DIR / "configs" / "config.yaml"
OBJECT_CONFIG_PATH = ROOT_DIR / "configs" / "object_config.yaml"
ATTRIBUTE_CONFIG_PATH = ROOT_DIR / "configs" / "attribute_config.yaml"
RELATION_CONFIG_PATH = ROOT_DIR / "configs" / "relation_config.yaml"

base_config = load_config(BASE_CONFIG_PATH)
object_config = load_task_configs(BASE_CONFIG_PATH, OBJECT_CONFIG_PATH)
attribute_config = load_task_configs(BASE_CONFIG_PATH, ATTRIBUTE_CONFIG_PATH)
relation_config = load_task_configs(BASE_CONFIG_PATH, RELATION_CONFIG_PATH)


def ns(**kwargs):
    return SimpleNamespace(**kwargs)


object_attribute_config = ns(
    backbone=ns(
        name=str(object_config.backbone.name),
        pretrained=bool(object_config.backbone.pretrained),
        learnable_backbone=bool(object_config.backbone.learnable_backbone),
        freeze_backbone=bool(object_config.backbone.freeze_backbone),
        freeze_epochs=int(object_config.backbone.freeze_epochs),
        feature_dim=int(object_config.backbone.feature_dim),
    ),
    dataset=ns(
        processed_dir=str(object_config.dataset.processed_dir),
        feature_cache_dir=str(object_config.dataset.feature_cache_dir),
        use_feature_cache=bool(object_config.dataset.use_feature_cache),
        object_input_mode=str(object_config.dataset.object_input_mode),
        attribute_input_mode=str(attribute_config.dataset.attribute_input_mode),
    ),
    labels=ns(
        max_objects=int(object_config.labels.max_objects),
        max_attributes=int(attribute_config.labels.max_attributes),
    ),
    model=ns(
        object_hidden_dim=int(object_config.model.object_hidden_dim),
        object_dropout=float(object_config.model.object_dropout),
        object_num_layers=int(object_config.model.object_num_layers),
        attribute_hidden_dim=int(attribute_config.model.attribute_hidden_dim),
        attribute_dropout=float(attribute_config.model.attribute_dropout),
        attribute_num_layers=int(attribute_config.model.attribute_num_layers),
    ),
    loss=ns(
        object_weight=float(object_config.loss.object_weight),
        attribute_weight=float(attribute_config.loss.attribute_weight),
        attribute_pos_weight=float(attribute_config.loss.attribute_pos_weight),
    ),
    augmentation=ns(
        random_horizontal_flip=float(object_config.augmentation.random_horizontal_flip),
        color_jitter=ns(
            enabled=bool(object_config.augmentation.color_jitter.enabled),
            brightness=float(object_config.augmentation.color_jitter.brightness),
            contrast=float(object_config.augmentation.color_jitter.contrast),
            saturation=float(object_config.augmentation.color_jitter.saturation),
            hue=float(object_config.augmentation.color_jitter.hue),
        ),
        random_erasing_p=float(object_config.augmentation.random_erasing_p),
        resize_delta=int(object_config.augmentation.resize_delta),
    ),
    eval=ns(
        object_top_k=list(object_config.eval.object_top_k),
        attribute_threshold_mode=str(attribute_config.eval.attribute_threshold_mode),
        attribute_threshold=float(attribute_config.eval.attribute_threshold),
        attribute_threshold_scale=float(attribute_config.eval.attribute_threshold_scale),
        attribute_threshold_min=float(attribute_config.eval.attribute_threshold_min),
        attribute_threshold_max=float(attribute_config.eval.attribute_threshold_max),
    ),
)

if torch.cuda.is_available() and str(base_config.device).startswith("cuda"):
    device = "cuda"
else:
    device = "cpu"

seed = int(base_config.seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

image_dir = ROOT_DIR / Path(base_config.dataset.image_dir)
checkpoint_root = ROOT_DIR / Path(base_config.paths.checkpoint_dir)

object_processed_dir = resolve_existing_path(
    ROOT_DIR / Path(object_config.dataset.processed_dir),
    ROOT_DIR / "data" / "processed_sample" / "task1",
)
attribute_processed_dir = resolve_existing_path(
    ROOT_DIR / Path(attribute_config.dataset.processed_dir),
    ROOT_DIR / "data" / "processed_sample" / "task1",
)
relation_processed_dir = resolve_existing_path(
    ROOT_DIR / Path(relation_config.dataset.processed_dir),
    ROOT_DIR / "data" / "processed_sample" / "task2",
)

object_vocab = load_vocab_file(object_processed_dir / "object_vocab.json")
attribute_vocab = load_vocab_file(attribute_processed_dir / "attribute_vocab.json")
relation_vocab = load_vocab_file(relation_processed_dir / "relation_vocab.json")

idx_to_object = inverse_vocab(object_vocab)
idx_to_attribute = inverse_vocab(attribute_vocab)
idx_to_relation = inverse_vocab(relation_vocab)

object_checkpoint = find_checkpoint_file(
    checkpoint_root,
    filename_prefixes=("object_", "task1_object_"),
)
attribute_checkpoint = find_checkpoint_file(
    checkpoint_root,
    filename_prefixes=("attribute_", "task1_attribute_"),
)
relation_checkpoint = find_checkpoint_file(
    checkpoint_root,
    filename_prefixes=("relation_", "task2_"),
)

if object_checkpoint is None:
    raise FileNotFoundError("Không tìm thấy checkpoint cho object pipeline.")
if attribute_checkpoint is None:
    raise FileNotFoundError("Không tìm thấy checkpoint cho attribute pipeline.")
if relation_checkpoint is None:
    raise FileNotFoundError("Không tìm thấy checkpoint cho relation pipeline.")

object_checkpoint_uses_backbone = checkpoint_uses_backbone(object_checkpoint)
attribute_checkpoint_uses_backbone = checkpoint_uses_backbone(attribute_checkpoint)
relation_checkpoint_uses_backbone = checkpoint_uses_backbone(relation_checkpoint)

object_learnable_backbone = object_checkpoint_uses_backbone
attribute_learnable_backbone = attribute_checkpoint_uses_backbone
relation_learnable_backbone = relation_checkpoint_uses_backbone

object_use_feature_cache = bool(object_config.dataset.use_feature_cache) and not object_learnable_backbone
attribute_use_feature_cache = bool(attribute_config.dataset.use_feature_cache) and not attribute_learnable_backbone
relation_use_feature_cache = bool(relation_config.dataset.use_feature_cache) and not relation_learnable_backbone

object_input_mode = str(object_config.dataset.object_input_mode)
attribute_input_mode = str(attribute_config.dataset.attribute_input_mode)
relation_input_mode = str(relation_config.dataset.input_mode)

object_model = ObjectClassifier(
    num_classes=len(object_vocab),
    feature_dim=int(object_config.backbone.feature_dim),
    hidden_dim=int(object_config.model.object_hidden_dim),
    dropout=float(object_config.model.object_dropout),
    num_layers=int(object_config.model.object_num_layers),
    backbone_name=str(object_config.backbone.name) if object_learnable_backbone else None,
    pretrained=bool(object_config.backbone.pretrained),
    freeze_backbone=bool(object_config.backbone.freeze_backbone),
    learnable_backbone=object_learnable_backbone,
    device=device,
)

attribute_model = AttributeClassifier(
    num_attributes=len(attribute_vocab),
    feature_dim=int(attribute_config.backbone.feature_dim),
    hidden_dim=int(attribute_config.model.attribute_hidden_dim),
    dropout=float(attribute_config.model.attribute_dropout),
    num_layers=int(attribute_config.model.attribute_num_layers),
    backbone_name=str(attribute_config.backbone.name) if attribute_learnable_backbone else None,
    pretrained=bool(attribute_config.backbone.pretrained),
    freeze_backbone=bool(attribute_config.backbone.freeze_backbone),
    learnable_backbone=attribute_learnable_backbone,
    device=device,
)

relation_model = RelationClassifier(
    num_relations=len(relation_vocab),
    feature_dim=int(relation_config.backbone.feature_dim),
    spatial_dim=int(relation_config.spatial.spatial_dim),
    hidden_dim=int(relation_config.model.hidden_dim),
    dropout=float(relation_config.model.dropout),
    num_layers=int(relation_config.model.num_layers),
    attention_heads=int(relation_config.model.attention_heads),
    fusion_method=str(relation_config.model.fusion_method),
    backbone_name=str(relation_config.backbone.name) if relation_learnable_backbone else None,
    pretrained=bool(relation_config.backbone.pretrained),
    freeze_backbone=bool(relation_config.backbone.freeze_backbone),
    learnable_backbone=relation_learnable_backbone,
    device=device,
)

e2e_model = VisualGenomeE2EModel.from_models(
    object_model=object_model,
    attribute_model=attribute_model,
    relation_model=relation_model,
)

e2e_model.eval()

print(f"Project root: {ROOT_DIR}")
print(f"Device: {device}")
print(f"Object processed dir: {object_processed_dir}")
print(f"Attribute processed dir: {attribute_processed_dir}")
print(f"Relation processed dir: {relation_processed_dir}")
print(f"Object checkpoint: {object_checkpoint}")
print(f"Attribute checkpoint: {attribute_checkpoint}")
print(f"Relation checkpoint: {relation_checkpoint}")
print(f"Runtime backbone mode - object: {object_learnable_backbone}, attribute: {attribute_learnable_backbone}, relation: {relation_learnable_backbone}")
print(f"Runtime cache mode - object: {object_use_feature_cache}, attribute: {attribute_use_feature_cache}, relation: {relation_use_feature_cache}")

loaded_checkpoints = e2e_model.load_checkpoints(
    object_checkpoint=str(object_checkpoint),
    attribute_checkpoint=str(attribute_checkpoint),
    relation_checkpoint=str(relation_checkpoint),
    strict=True,
)

print("Loaded checkpoint metadata:")
print(loaded_checkpoints)
print("E2E summary:")
print(e2e_model.save_summary())

caption_generator = CaptionGenerator(
    object_vocab=object_vocab,
    attribute_vocab=attribute_vocab,
    relation_vocab=relation_vocab,
)

print("Caption generator ready.")

Resolved project root: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
Sys path head: ['C:\\Users\\caoha\\OneDrive\\Desktop\\study\\Ky2\\CV\\prj', 'c:\\Users\\caoha\\anaconda3\\python313.zip', 'c:\\Users\\caoha\\anaconda3\\DLLs']
Src exists: True | Data package exists: True
Project root: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj
Device: cpu
Object/attribute processed dir: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1
Relation processed dir: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task2
Object checkpoint: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\checkpoints\task1_object\task1_object_epoch_32.pth
Attribute checkpoint: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\checkpoints\task1_attribute\task1_attribute_epoch_27.pth
Relation checkpoint: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\checkpoints\task2\task2_epoch_29.pth
Runtime backbone mode - object: False, attribute: False, relation: False
Runtime cach

In [ ]:
image_mean = list(base_config.image.mean)
image_std = list(base_config.image.std)

_, _, object_test_dataset = build_object_datasets(
    processed_dir=str(object_processed_dir),
    image_dir=str(image_dir),
    roi_size=int(base_config.image.roi_size),
    use_feature_cache=object_use_feature_cache,
    feature_cache_dir=str(object_processed_dir / object_config.dataset.feature_cache_dir),
    input_mode=object_input_mode,
    max_samples=1,
    train_horizontal_flip_p=float(object_config.augmentation.random_horizontal_flip),
    train_color_jitter=bool(object_config.augmentation.color_jitter.enabled),
    train_brightness=float(object_config.augmentation.color_jitter.brightness),
    train_contrast=float(object_config.augmentation.color_jitter.contrast),
    train_saturation=float(object_config.augmentation.color_jitter.saturation),
    train_hue=float(object_config.augmentation.color_jitter.hue),
    train_random_erasing_p=float(object_config.augmentation.random_erasing_p),
    train_resize_delta=int(object_config.augmentation.resize_delta),
    mean=image_mean,
    std=image_std,
)

_, _, attribute_test_dataset = build_attribute_datasets(
    processed_dir=str(attribute_processed_dir),
    image_dir=str(image_dir),
    roi_size=int(base_config.image.roi_size),
    use_feature_cache=attribute_use_feature_cache,
    feature_cache_dir=str(attribute_processed_dir / attribute_config.dataset.feature_cache_dir),
    input_mode=attribute_input_mode,
    max_samples=1,
    train_horizontal_flip_p=float(attribute_config.augmentation.random_horizontal_flip),
    train_color_jitter=bool(attribute_config.augmentation.color_jitter.enabled),
    train_brightness=float(attribute_config.augmentation.color_jitter.brightness),
    train_contrast=float(attribute_config.augmentation.color_jitter.contrast),
    train_saturation=float(attribute_config.augmentation.color_jitter.saturation),
    train_hue=float(attribute_config.augmentation.color_jitter.hue),
    train_random_erasing_p=float(attribute_config.augmentation.random_erasing_p),
    train_resize_delta=int(attribute_config.augmentation.resize_delta),
    mean=image_mean,
    std=image_std,
)

_, _, relation_test_dataset = build_relation_datasets(
    processed_dir=str(relation_processed_dir),
    image_dir=str(image_dir),
    roi_size=int(base_config.image.roi_size),
    use_feature_cache=relation_use_feature_cache,
    use_spatial_features=bool(relation_config.spatial.use_spatial_features),
    feature_cache_dir=str(relation_processed_dir / relation_config.dataset.feature_cache_dir),
    input_mode=relation_input_mode,
    max_samples=1,
    train_horizontal_flip_p=float(relation_config.augmentation.random_horizontal_flip),
    train_color_jitter=bool(relation_config.augmentation.color_jitter.enabled),
    train_brightness=float(relation_config.augmentation.color_jitter.brightness),
    train_contrast=float(relation_config.augmentation.color_jitter.contrast),
    train_saturation=float(relation_config.augmentation.color_jitter.saturation),
    train_hue=float(relation_config.augmentation.color_jitter.hue),
    train_random_erasing_p=float(relation_config.augmentation.random_erasing_p),
    train_resize_delta=int(relation_config.augmentation.resize_delta),
    mean=image_mean,
    std=image_std,
)

if len(object_test_dataset) == 0:
    raise RuntimeError('Object test dataset rỗng.')
if len(attribute_test_dataset) == 0:
    raise RuntimeError('Attribute test dataset rỗng.')
if len(relation_test_dataset) == 0:
    raise RuntimeError('Relation test dataset rỗng.')

object_sample = object_test_dataset[0]
attribute_sample = attribute_test_dataset[0]
relation_sample = relation_test_dataset[0]

object_inputs = to_batch_tensor(object_sample, ('feature', 'image'), device)
attribute_inputs = to_batch_tensor(attribute_sample, ('feature', 'image'), device)
relation_inputs = to_batch_tensor(relation_sample, ('feature', 'image'), device)
relation_spatial = relation_sample['spatial'].unsqueeze(0).to(device)

with torch.no_grad():
    object_logits = object_model(object_inputs)
    attribute_logits = attribute_model(attribute_inputs)
    relation_logits = relation_model(relation_inputs, relation_spatial)

print('Object sample meta:')
print(object_sample.get('meta', {}))
print('Attribute sample meta:')
print(attribute_sample.get('meta', {}))
print('Relation sample meta:')
print(relation_sample.get('meta', {}))

print('Top object predictions:')
print(top_class_predictions(object_logits, idx_to_object, top_k=5))
print('Top attribute predictions:')
print(top_attribute_predictions(attribute_logits, idx_to_attribute, top_k=5))
print('Top relation predictions:')
print(top_class_predictions(relation_logits, idx_to_relation, top_k=5))

object_attribute_results = {
    'object_logits': object_logits.cpu(),
    'attribute_logits': attribute_logits.cpu(),
}
relation_results = {
    'relation_logits': relation_logits.cpu(),
    'subject_name': relation_sample.get('meta', {}).get('subject_name', ''),
    'object_name': relation_sample.get('meta', {}).get('object_name', ''),
}

captions = caption_generator.generate_from_predictions(
    object_attribute_results=object_attribute_results,
    relation_results=relation_results,
    top_k=3,
)

print('Generated captions:')
for caption in captions:
    print(f'- {caption}')

cleanup_cuda_memory(note='E2E wrapper demo finished')

[ObjectAttributeDataset] Loaded 4494 annotations từ C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\train\annotations.json
[ObjectAttributeDataset] Loading feature cache: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\features\train_features.pt
[ObjectAttributeDataset] Loaded 4494 cached features
[ObjectAttributeDataset] Giới hạn 1 samples (debug mode)
ObjectAttributeDataset [train]: 1 ROIs | 1 images | 301 objects | 201 attributes | cache=✅
[ObjectAttributeDataset] Loaded 444 annotations từ C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\val\annotations.json
[ObjectAttributeDataset] Loading feature cache: C:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed_sample\task1\features\val_features.pt
[ObjectAttributeDataset] Loaded 444 cached features
[ObjectAttributeDataset] Giới hạn 1 samples (debug mode)
ObjectAttributeDataset [val]: 1 ROIs | 1 images | 301 objects | 201 attributes | cache=